In [ ]:
import sys
import subprocess
import os

# Check which Python is running
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")

# Check pip list from within notebook
result = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
print(f"\nPip list from this Python:\n{result.stdout}")

# Check transformers specifically
import transformers
print(f"\nTransformers version: {transformers.__version__}")
print(f"Transformers location: {transformers.__file__}")

In [2]:
import sys 
sys.path.append('/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/qwen_tts/')
sys.path.append('/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/qwen_tts/core/tokenizer_12hz/')

In [4]:
# testing this tokenizer for some audio files and seeing reconstruction 

SPEECH_CONFIG_DIR = '/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/Qwen3-TTS-12Hz-1.7B-VoiceDesign/speech_tokenizer/'
TOKENIZER_WEIGHTS = '/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/Qwen3-TTS-12Hz-1.7B-VoiceDesign/speech_tokenizer/model.safetensors'


# architecture files for tokenizer 
import sys
sys.path.append("/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS")

from qwen_tts.core.tokenizer_12hz.modeling_qwen3_tts_tokenizer_v2 import (
    Qwen3TTSTokenizerV2Model,
    Qwen3TTSTokenizerV2PreTrainedModel,
) # we have added it like this so that the relative imports are 
# Qwen3TTSTokenizerV2PreTrainedModel()
# update the transformer .venv file 


********
********
 
4.57.3
/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/.venv/lib/python3.14/site-packages/transformers/__init__.py


In [7]:
SPEECH_CONFIG_DIR = "/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/Qwen3-TTS-12Hz-1.7B-VoiceDesign/speech_tokenizer/"

from qwen_tts.core.tokenizer_12hz.configuration_qwen3_tts_tokenizer_v2 import Qwen3TTSTokenizerV2Config
from qwen_tts.core.tokenizer_12hz.modeling_qwen3_tts_tokenizer_v2 import Qwen3TTSTokenizerV2Model

config = Qwen3TTSTokenizerV2Config.from_pretrained(SPEECH_CONFIG_DIR)
print(config.decoder_config.codebook_dim)  # should be 512

model = Qwen3TTSTokenizerV2Model.from_pretrained(SPEECH_CONFIG_DIR)

512


In [10]:
from huggingface_hub import load_state_dict_from_file
weights = load_state_dict_from_file(TOKENIZER_WEIGHTS)

model.load_state_dict(weights,strict = True)

<All keys matched successfully>

In [28]:
model.eval()
from torch.nn.utils.rnn import pad_sequence

AUDIO_FILE = "/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/testing/output.wav"
REGENERATED_AUDIO_FILE = "/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/testing/regenerated.wav"
import librosa

audio, sr = librosa.load(AUDIO_FILE, sr = None)
print("Current sample rate : ", sr)

import torch
audio = torch.from_numpy(audio).unsqueeze(0)
padding_mask = torch.ones(audio.shape, dtype=torch.bool)  # (1, T), all valid
encoded = model.encode(audio, padding_mask)
codes = pad_sequence(encoded.audio_codes, batch_first=True, padding_value=-1)

# print(encoded.shape)

decoded = model.decode(codes)
wav = decoded.audio_values[0].cpu().detach().numpy()
out_sr = model.get_output_sample_rate()  # 24000
import soundfile as sf
sf.write(REGENERATED_AUDIO_FILE, wav, out_sr)


Current sample rate :  24000


In [19]:
import librosa
import torch
import soundfile as sf
from torch.nn.utils.rnn import pad_sequence

AUDIO_FILE = "/Users/mohitdulani/Desktop/personal/audio-models/Qwen3-TTS/testing/output.wav"
TARGET_SR = model.get_input_sample_rate()  # 24000

audio, sr = librosa.load(AUDIO_FILE, sr=TARGET_SR, mono=True)
audio = torch.from_numpy(audio).float().unsqueeze(0)  # (1, T)

padding_mask = torch.ones_like(audio)  # (1, T), all valid

with torch.inference_mode():
    enc = model.encode(audio, padding_mask)
    codes = pad_sequence(enc.audio_codes, batch_first=True, padding_value=-1)
    dec = model.decode(codes)

out_sr = model.get_output_sample_rate()  # 24000
sf.write(REGENERATED_AUDIO_FILE, dec.audio_values[0].cpu().numpy(), out_sr)

TypeError: only integer tensors of a single element can be converted to an index